# OT (Offensive Tackle) round regression (Ridge)

Predict draft round 1–8 (8 = undrafted) using combine + PFF (Pass_Blocking, Run_Blocking) + RAS, arm length, KNN imputation, Ridge regression.
- Train: 2015–2023 (ot_training.csv; RAS and PFF already merged in data_cleaning).
- Test: ot_testing.csv filtered to 2024/2025 (drafted only); 2026 from ot_drafted_2026.csv.

In [140]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# OT feature set: athletic + PFF derived rates (data_cleaning.py) + penalty + p4
# PFF derived: true_pass_set_pressure_rate = pressures_allowed/snap_counts_pass_block; true_pass_set_sack_rate = sacks_allowed/snap_counts_pass_block
OT_FEATURES_WITH_COLLEGE = [
    'Height', 'Weight', 'arm_length_inches', 'arm_33_plus', 'arm_34_plus', 'speed_score', 'agility_score',
    'true_pass_set_pressure_rate', 'true_pass_set_sack_rate', 'snap_counts_pass_block',
    'snap_counts_run_block', 'grades_run_block', 'gap_rate', 'zone_rate', 'penalty_rate', 'p4_conference'
]
CONTAINS_WITH_COLLEGE_OT = [
    'contains_height', 'contains_weight', 'contains_arm_length_inches', 'contains_speed_score', 'contains_agility_score',
    'contains_true_pass_set_pressure_rate', 'contains_true_pass_set_sack_rate', 'contains_snap_counts_pass_block',
    'contains_snap_counts_run_block', 'contains_grades_run_block', 'contains_gap_rate', 'contains_zone_rate', 'contains_penalty_rate', 'contains_p4_conference'
]
FEATURES_WITH_COLLEGE_ALL = OT_FEATURES_WITH_COLLEGE + CONTAINS_WITH_COLLEGE_OT

In [141]:
# Load OT training (2015–2023)
df = pd.read_csv('../data/processed/ot_training.csv')
df = df[df['Year'].between(2015, 2023)].copy()
print('Train (2015–2023 OTs):', len(df))

Train (2015–2023 OTs): 190


In [142]:
# Print RAS, PFF, and arm length availability
total_count = len(df)
ras_count = df['RAS'].notna().sum()
print(f"Players with RAS: {ras_count} out of {total_count} ({ras_count/total_count*100:.1f}%)")
print(f"Players with true_pass_set_pressure_rate: {df['true_pass_set_pressure_rate'].notna().sum()} out of {total_count}")
print(f"Players with snap_counts_run_block: {df['snap_counts_run_block'].notna().sum()} out of {total_count}")
print(f"Players with arm_length_inches: {df['arm_length_inches'].notna().sum()} out of {total_count}")

Players with RAS: 80 out of 190 (42.1%)
Players with true_pass_set_pressure_rate: 149 out of 190
Players with snap_counts_run_block: 149 out of 190
Players with arm_length_inches: 187 out of 190


In [143]:
def height_inches(h):
    if pd.isna(h): return np.nan
    if isinstance(h, (int, float)) and not (isinstance(h, float) and np.isnan(h)):
        return float(h)
    s = str(h).strip()
    if '-' in s:
        parts = s.split('-')
        return int(parts[0]) * 12 + int(parts[1])
    return np.nan
df['Height'] = df['Height'].apply(height_inches)

df['speed_score'] = np.where(
    df['40yd'].notna() & (df['40yd'] > 0),
    df['Weight'] * 200 / (df['40yd'] ** 4),
    np.nan
)

# Agility score: lower 3Cone/Shuttle = better; z-scores from training, then -(z_3cone + z_shuttle)
mean_3c = df['3Cone'].mean()
std_3c = df['3Cone'].std()
mean_sh = df['Shuttle'].mean()
std_sh = df['Shuttle'].std()
if std_3c == 0 or np.isnan(std_3c): std_3c = 1.0
if std_sh == 0 or np.isnan(std_sh): std_sh = 1.0
z_3 = (df['3Cone'] - mean_3c) / std_3c
z_sh = (df['Shuttle'] - mean_sh) / std_sh
df['agility_score'] = (-z_3.fillna(0)) + (-z_sh.fillna(0))

school_alias = {'Ole Miss': 'Mississippi', 'Miami (FL)': 'Miami', 'Southern California': 'USC', 'Ohio St.': 'Ohio State',
    'Florida St.': 'Florida State', 'Penn St.': 'Penn State', 'NC State': 'North Carolina State', 'Oregon St.': 'Oregon State'}
SEC_SCHOOLS = {'Alabama', 'Arkansas', 'Auburn', 'Florida', 'Georgia', 'Kentucky', 'LSU', 'Mississippi', 'Mississippi State', 'Missouri', 'South Carolina', 'Tennessee', 'Texas A&M', 'Vanderbilt', 'Oklahoma', 'Texas'}
BIG_TEN_SCHOOLS = {'Illinois', 'Indiana', 'Iowa', 'Maryland', 'Michigan', 'Michigan State', 'Minnesota', 'Nebraska', 'Northwestern', 'Ohio State', 'Penn State', 'Purdue', 'Rutgers', 'Wisconsin', 'UCLA', 'USC', 'Oregon', 'Washington'}
BIG_12_SCHOOLS = {'Baylor', 'Iowa State', 'Kansas', 'Kansas State', 'Oklahoma State', 'TCU', 'Texas Tech', 'West Virginia', 'BYU', 'UCF', 'Cincinnati', 'Houston', 'Arizona', 'Arizona State', 'Colorado', 'Utah'}
ACC_SCHOOLS = {'Boston College', 'Clemson', 'Duke', 'Florida State', 'Georgia Tech', 'Louisville', 'Miami', 'North Carolina', 'North Carolina State', 'NC State', 'Pittsburgh', 'Syracuse', 'Virginia', 'Virginia Tech', 'Wake Forest', 'California', 'SMU', 'Stanford'}
PAC12_SCHOOLS = {'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon', 'Oregon State', 'Stanford', 'UCLA', 'USC', 'Utah', 'Washington', 'Washington State'}
P4_SCHOOLS = SEC_SCHOOLS | BIG_TEN_SCHOOLS | BIG_12_SCHOOLS | ACC_SCHOOLS | PAC12_SCHOOLS
P4_SCHOOLS_NO_PAC12 = SEC_SCHOOLS | BIG_TEN_SCHOOLS | BIG_12_SCHOOLS | ACC_SCHOOLS

def is_p4(row):
    s = row.get('School')
    if pd.isna(s) or s == '': return 0
    sn = school_alias.get(s, s)
    year = row.get('Year', 2023)
    schools = P4_SCHOOLS if year <= 2023 else P4_SCHOOLS_NO_PAC12
    return 1 if sn in schools else 0
df['p4_conference'] = df.apply(is_p4, axis=1)

df['contains_height'] = df['Height'].notna().astype(int)
df['contains_weight'] = df['Weight'].notna().astype(int)
df['contains_arm_length_inches'] = df['arm_length_inches'].notna().astype(int) if 'arm_length_inches' in df.columns else 0
df['arm_33_plus'] = (df['arm_length_inches'] >= 33).fillna(False).astype(int) if 'arm_length_inches' in df.columns else 0
df['arm_34_plus'] = (df['arm_length_inches'] >= 34).fillna(False).astype(int) if 'arm_length_inches' in df.columns else 0
df['contains_speed_score'] = df['speed_score'].notna().astype(int)
df['contains_agility_score'] = (df['3Cone'].notna() | df['Shuttle'].notna()).astype(int)
df['contains_ras'] = df['RAS'].notna().astype(int)
df['contains_true_pass_set_pressure_rate'] = df['true_pass_set_pressure_rate'].notna().astype(int) if 'true_pass_set_pressure_rate' in df.columns else 0
df['contains_true_pass_set_sack_rate'] = df['true_pass_set_sack_rate'].notna().astype(int) if 'true_pass_set_sack_rate' in df.columns else 0
df['contains_snap_counts_pass_block'] = df['snap_counts_pass_block'].notna().astype(int) if 'snap_counts_pass_block' in df.columns else 0
df['contains_snap_counts_run_block'] = df['snap_counts_run_block'].notna().astype(int) if 'snap_counts_run_block' in df.columns else 0
df['contains_grades_run_block'] = df['grades_run_block'].notna().astype(int) if 'grades_run_block' in df.columns else 0
df['contains_gap_rate'] = df['gap_rate'].notna().astype(int) if 'gap_rate' in df.columns else 0
df['contains_zone_rate'] = df['zone_rate'].notna().astype(int) if 'zone_rate' in df.columns else 0
df['contains_penalty_rate'] = df['penalty_rate'].notna().astype(int) if 'penalty_rate' in df.columns else 0
df['contains_p4_conference'] = df['School'].notna().astype(int)

In [144]:
y = np.where(df['Drafted'].astype(bool), np.clip(df['Round'].fillna(1).astype(int), 1, 7), 8)
X_raw = df[FEATURES_WITH_COLLEGE_ALL].copy()
imputer = KNNImputer(n_neighbors=10)
X = imputer.fit_transform(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_scaled, y)
y_pred_train = np.clip(ridge.predict(X_scaled), 1, 8)
print('Train MAE (round 1–8):', round(mean_absolute_error(y, y_pred_train), 4))
print('Train samples:', len(y))

Train MAE (round 1–8): 1.7813
Train samples: 190


In [145]:
def prepare_ot_df(ldf, year):
    ldf = ldf.copy()
    ldf['Year'] = year
    for col in ['Height', 'Weight', '40yd', 'Vertical', 'Bench', 'Broad Jump', '3Cone', 'Shuttle', 'RAS'] + OT_FEATURES_WITH_COLLEGE:
        if col not in ldf.columns:
            ldf[col] = np.nan
    if ldf['Height'].dtype == object or (ldf['Height'].astype(str).str.contains('-', na=False).any()):
        ldf['Height'] = ldf['Height'].apply(height_inches)
    else:
        ldf['Height'] = pd.to_numeric(ldf['Height'], errors='coerce')
    ldf['speed_score'] = np.where(ldf['40yd'].notna() & (ldf['40yd'] > 0), ldf['Weight'] * 200 / (ldf['40yd'] ** 4), np.nan)
    z_3 = (ldf['3Cone'] - mean_3c) / std_3c
    z_sh = (ldf['Shuttle'] - mean_sh) / std_sh
    ldf['agility_score'] = (-z_3.fillna(0)) + (-z_sh.fillna(0))
    ldf['p4_conference'] = ldf.apply(is_p4, axis=1)
    ldf['contains_height'] = ldf['Height'].notna().astype(int)
    ldf['contains_weight'] = ldf['Weight'].notna().astype(int)
    ldf['contains_arm_length_inches'] = ldf['arm_length_inches'].notna().astype(int) if 'arm_length_inches' in ldf.columns else 0
    ldf['arm_33_plus'] = (ldf['arm_length_inches'] >= 33).fillna(False).astype(int) if 'arm_length_inches' in ldf.columns else 0
    ldf['arm_34_plus'] = (ldf['arm_length_inches'] >= 34).fillna(False).astype(int) if 'arm_length_inches' in ldf.columns else 0
    ldf['contains_speed_score'] = ldf['speed_score'].notna().astype(int)
    ldf['contains_agility_score'] = (ldf['3Cone'].notna() | ldf['Shuttle'].notna()).astype(int)
    ldf['contains_ras'] = ldf['RAS'].notna().astype(int)
    ldf['contains_true_pass_set_pressure_rate'] = ldf['true_pass_set_pressure_rate'].notna().astype(int) if 'true_pass_set_pressure_rate' in ldf.columns else 0
    ldf['contains_true_pass_set_sack_rate'] = ldf['true_pass_set_sack_rate'].notna().astype(int) if 'true_pass_set_sack_rate' in ldf.columns else 0
    ldf['contains_snap_counts_pass_block'] = ldf['snap_counts_pass_block'].notna().astype(int) if 'snap_counts_pass_block' in ldf.columns else 0
    ldf['contains_snap_counts_run_block'] = ldf['snap_counts_run_block'].notna().astype(int) if 'snap_counts_run_block' in ldf.columns else 0
    ldf['contains_grades_run_block'] = ldf['grades_run_block'].notna().astype(int) if 'grades_run_block' in ldf.columns else 0
    ldf['contains_gap_rate'] = ldf['gap_rate'].notna().astype(int) if 'gap_rate' in ldf.columns else 0
    ldf['contains_zone_rate'] = ldf['zone_rate'].notna().astype(int) if 'zone_rate' in ldf.columns else 0
    ldf['contains_penalty_rate'] = ldf['penalty_rate'].notna().astype(int) if 'penalty_rate' in ldf.columns else 0
    ldf['contains_p4_conference'] = ldf['School'].notna().astype(int)
    return ldf

ot_testing = pd.read_csv('../data/processed/ot_testing.csv')
ot_2024 = prepare_ot_df(ot_testing[ot_testing['Year'] == 2024], 2024)
ot_2025 = prepare_ot_df(ot_testing[ot_testing['Year'] == 2025], 2025)
X_24_raw = ot_2024[FEATURES_WITH_COLLEGE_ALL].copy()
X_25_raw = ot_2025[FEATURES_WITH_COLLEGE_ALL].copy()
X_24 = imputer.transform(X_24_raw)
X_25 = imputer.transform(X_25_raw)
X_24_scaled = scaler.transform(X_24)
X_25_scaled = scaler.transform(X_25)
pred_24 = np.clip(ridge.predict(X_24_scaled), 1, 8)
pred_25 = np.clip(ridge.predict(X_25_scaled), 1, 8)
actual_24 = ot_2024['Round'].astype(int).values
actual_25 = ot_2025['Round'].astype(int).values

def eval_metrics(actual, pred, label):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    exact = (np.round(pred) == actual).mean()
    within_1 = (np.abs(np.round(pred) - actual) <= 1).mean()
    print(f'{label} (n={len(actual)}): MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, Exact={exact:.2%}, Within-1={within_1:.2%}')

print('2024 OTs:')
eval_metrics(actual_24, pred_24, '2024')
print('2025 OTs:')
eval_metrics(actual_25, pred_25, '2025')

2024 OTs:
2024 (n=28): MAE=2.0175, RMSE=2.2307, R²=0.1166, Exact=3.57%, Within-1=39.29%
2025 OTs:
2025 (n=18): MAE=1.3931, RMSE=1.8360, R²=0.3580, Exact=27.78%, Within-1=55.56%


In [146]:
def pred_round_to_tier(p):
    if p < 1.5: return ('1st', 'Elite')
    if p < 2.5: return ('2nd', 'Day 2')
    if p < 4.5: return ('3rd-4th', 'Day 2')
    if p < 6.5: return ('5th-6th', 'Day 3')
    if p < 7.5: return ('7th', 'Day 3')
    return ('UDFA', 'Undrafted')

ot_2024_display = ot_2024[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
ot_2024_display['predicted_round'] = pred_24
ot_2024_display['tier_label'] = [pred_round_to_tier(x)[0] for x in pred_24]
ot_2025_display = ot_2025[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
ot_2025_display['predicted_round'] = pred_25
ot_2025_display['tier_label'] = [pred_round_to_tier(x)[0] for x in pred_25]
print('2024 drafted OTs')
display(ot_2024_display.sort_values('predicted_round'))
print('2025 drafted OTs')
display(ot_2025_display.sort_values('predicted_round'))

2024 drafted OTs


,Round,Pick,Player,School,Year,predicted_round,tier_label
0,1.0,6.0,Joe Alt,NOTRE DAME,2024,1.469693,1st
5,1.0,18.0,Amarius Mims,GEORGIA,2024,2.065011,2nd
4,1.0,14.0,Taliese Fuaga,OREGON ST,2024,2.301282,2nd
1,1.0,7.0,J.C. Latham,ALABAMA,2024,3.255387,3rd-4th
6,2.0,55.0,Patrick Paul,HOUSTON,2024,3.336243,3rd-4th
18,6.0,162.0,Christian Jones,TEXAS,2024,3.627893,3rd-4th
20,7.0,233.0,Nathan Thomas,LA LAFAYET,2024,3.771093,3rd-4th
21,7.0,204.0,Tylan Grable,UCF,2024,3.822832,3rd-4th
15,3.0,67.0,Brandon Coleman,TCU,2024,3.830270,3rd-4th
2,1.0,11.0,Olumuyiwa Fashanu,PENN STATE,2024,4.153026,3rd-4th


2025 drafted OTs


,Round,Pick,Player,School,Year,predicted_round,tier_label
29,1.0,7.0,Armand Membou,Missouri,2025,1.000000,1st
34,2.0,48.0,Aireontae Ersery,Minnesota,2025,2.013857,2nd
35,2.0,37.0,Jonah Savaiinaea,Arizona,2025,2.208771,2nd
30,1.0,9.0,Kelvin Banks Jr,Texas,2025,2.665491,3rd-4th
31,1.0,32.0,Josh Simmons,Ohio State,2025,3.206090,3rd-4th
32,1.0,29.0,Josh Conerly Jr,Oregon,2025,3.422332,3rd-4th
42,7.0,250.0,John Williams,Cincinnati,2025,3.929486,3rd-4th
41,7.0,218.0,Jack Nelson,Wisconsin,2025,4.122419,3rd-4th
28,1.0,4.0,Will Campbell,LSU,2025,4.149354,3rd-4th
43,5.0,141.0,Carson Vinson,Alabama A&M,2025,4.487856,3rd-4th


In [147]:
# 2026 from ot_drafted_2026.csv
ot_2026 = prepare_ot_df(pd.read_csv('ot_drafted_2026.csv'), 2026)
if len(ot_2026) == 0:
    print('No 2026 OTs in ot_drafted_2026.csv')
else:
    X_26_raw = ot_2026[FEATURES_WITH_COLLEGE_ALL].copy()
    X_26 = imputer.transform(X_26_raw)
    X_26_scaled = scaler.transform(X_26)
    pred_26 = np.clip(ridge.predict(X_26_scaled), 1, 8)
    ot_2026_display = ot_2026[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
    ot_2026_display['predicted_round'] = pred_26
    ot_2026_display['tier_label'] = [pred_round_to_tier(x)[0] for x in pred_26]
    ot_2026_display[['Player','School','predicted_round']].assign(Pos='OT').to_csv('../data/processed/ot_2026_predictions.csv', index=False)
    print(f'2026 OTs (n={len(pred_26)}): Predictions generated')
    display(ot_2026_display.sort_values('predicted_round'))

2026 OTs (n=24): Predictions generated


,Round,Pick,Player,School,Year,predicted_round,tier_label
2,NaN,NaN,Kadyn Proctor,Alabama,2026,1.215362,1st
1,NaN,NaN,Spencer Fano,Utah,2026,2.818533,3rd-4th
10,NaN,NaN,Austin Barber,Florida,2026,3.898232,3rd-4th
6,NaN,NaN,Blake Miller,Clemson,2026,4.444737,3rd-4th
12,NaN,NaN,Jude Bowry,Boston College,2026,4.559470,5th-6th
14,NaN,NaN,Kage Casey,Boise State,2026,4.811580,5th-6th
8,NaN,NaN,Max Iheanachor,Arizona State,2026,4.822664,5th-6th
0,NaN,NaN,Francis Mauigoa,Miami,2026,4.935649,5th-6th
13,NaN,NaN,J.C. Davis,New Mexico,2026,5.058781,5th-6th
4,NaN,NaN,Monroe Freeling,Georgia,2026,5.134583,5th-6th


In [148]:
# Model results on entire training set (2017–2023), ordered by predicted_round
train_display = df[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
train_display['predicted_round'] = y_pred_train
train_display['tier_label'] = [pred_round_to_tier(x)[0] for x in y_pred_train]
train_display['interpretation'] = [pred_round_to_tier(x)[1] for x in y_pred_train]
train_display = train_display.sort_values('predicted_round').reset_index(drop=True)
train_display

,Round,Pick,Player,School,Year,predicted_round,tier_label,interpretation
0,4.0,111.0,Dawand Jones,Ohio St.,2023,1.000000,1st,Elite
1,1.0,22.0,Andre Dillard,Washington State,2019,1.000000,1st,Elite
2,1.0,8.0,Jack Conklin,Michigan State,2016,1.299468,1st,Elite
3,1.0,24.0,Tyler Smith,Tulsa,2022,1.527642,2nd,Day 2
4,1.0,6.0,Ikem Ekwonu,North Carolina St.,2022,1.556355,2nd,Day 2
...,...,...,...,...,...,...,...,...
185,3.0,94.0,Alex Cappa,Humboldt State,2018,7.752549,UDFA,Undrafted
186,NaN,NaN,Andre James,UCLA,2019,7.796637,UDFA,Undrafted
187,NaN,NaN,Brett Toth,Army,2018,8.000000,UDFA,Undrafted
188,NaN,NaN,Myron Cunningham,Arkansas,2022,8.000000,UDFA,Undrafted
